# Enrich graph with synthetic movie viewership

This notebook generates fake but statistically consistent movie viewership for customers subscribed to `streaming_movies`, then writes the data back to Neo4j.

### What it creates
- `:Movie` nodes with properties: `title`, `duration`, `year`, `rating`
- `(:Customer)-[:WATCHED_MOVIE {stream_date, streamed_for, event_id}]->(:Movie)`
- `(:Customer)-[:RATED {rating}]->(:Movie)` for roughly 5-10% of customers

### Statistical assumptions
- Most customers stream between 1 and 3 movies per week
- Timeline length is based on customer tenure (`tenure_month` fallback to `tenure_months`)
- Streamed minutes are bounded by movie duration and centered around partial-to-full completion

In [1]:
from dotenv import load_dotenv
from neo4j import GraphDatabase
import pandas as pd
import numpy as np
from datetime import date, timedelta
from tqdm.auto import tqdm
import os
import math

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
ENABLE_CLEANUP = os.getenv("ENABLE_CLEANUP", "False").lower() == "true"

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("Connected to Neo4j")

Connected to Neo4j


In [2]:
query_customers = """
MATCH (c:Customer)-[:SUBSCRIBES_TO]->(:Service {service_type: 'streaming_movies'})
RETURN c.customer_id AS customer_id,
       toInteger(coalesce(c.tenure_month, c.tenure_months, 1)) AS tenure_months
"""

with driver.session(database=NEO4J_DATABASE) as session:
    customer_rows = [r.data() for r in session.run(query_customers)]

customers_df = pd.DataFrame(customer_rows)
if customers_df.empty:
    raise ValueError("No customers found with 'streaming_movies' service.")

customers_df["tenure_months"] = (
    customers_df["tenure_months"].clip(lower=1).fillna(1).astype(int)
)
customers_df.head(), len(customers_df)

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `tenure_month` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=4, column=29, offset=149>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 149, 'line': 4, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (c:Customer)-[:SUBSCRIBES_TO]->(:Service {service_type: 'streaming_movies'})\nRETURN c.customer_id AS customer_id,\n       toInteger(coalesce(c.tenure_month, c.tenure_months, 1)) AS tenure_months\n"


(  customer_id  tenure_months
 0  8640-SDGKB              4
 1  4710-FDUIZ             56
 2  7721-JXEAW             59
 3  5673-FSSMF              1
 4  0447-BEMNG             48,
 2732)

In [3]:
adjectives = [
    "Silent",
    "Neon",
    "Hidden",
    "Midnight",
    "Golden",
    "Broken",
    "Crimson",
    "Echo",
    "Fading",
    "Electric",
    "Dark",
    "Bright",
    "Velvet",
    "Wild",
    "Last",
    "First",
    "Lost",
]
nouns = [
    "Signal",
    "Orbit",
    "City",
    "Horizon",
    "Archive",
    "Memory",
    "Voyage",
    "Code",
    "Storm",
    "Promise",
    "Shadow",
    "Bridge",
    "Frontier",
    "Pulse",
    "Beacon",
    "Frame",
]

N_MOVIES = 400
titles = set()
while len(titles) < N_MOVIES:
    base = f"{rng.choice(adjectives)} {rng.choice(nouns)}"
    if rng.random() < 0.22:
        base = f"{base} {rng.integers(2, 6)}"
    titles.add(base)

movies_df = pd.DataFrame(
    {
        "title": sorted(list(titles)),
    }
)
movies_df["duration"] = rng.integers(80, 181, size=len(movies_df))
movies_df["year"] = rng.integers(1995, date.today().year + 1, size=len(movies_df))
movies_df["rating"] = np.round(
    np.clip(rng.normal(3.6, 0.7, size=len(movies_df)), 1.0, 5.0), 1
)

movies_df.head()

,title,duration,year,rating
0,Bright Archive,128,2018,3.1
1,Bright Beacon,104,2018,2.8
2,Bright Bridge,159,2023,2.1
3,Bright Bridge 4,138,2004,3.4
4,Bright City,155,2001,3.2


In [4]:
# Weekly stream count distribution: most mass on 1-3 streams/week
weekly_choices = np.array([0, 1, 2, 3, 4])
weekly_probs = np.array([0.08, 0.34, 0.36, 0.18, 0.04])

today = date.today()
movie_titles = movies_df["title"].to_numpy()
movie_duration_map = dict(zip(movies_df["title"], movies_df["duration"]))

watch_rows = []

for row in customers_df.itertuples(index=False):
    customer_id = row.customer_id
    tenure_months = max(1, int(row.tenure_months))
    tenure_weeks = max(1, int(math.ceil(tenure_months * 4.345)))

    start_date = today - timedelta(weeks=tenure_weeks)
    streams_per_week = rng.choice(weekly_choices, size=tenure_weeks, p=weekly_probs)

    for w, n_streams in enumerate(streams_per_week):
        if n_streams == 0:
            continue

        week_start = start_date + timedelta(weeks=int(w))
        day_offsets = rng.integers(0, 7, size=int(n_streams))

        for day_offset in day_offsets:
            stream_dt = week_start + timedelta(days=int(day_offset))
            if stream_dt > today:
                continue

            title = str(rng.choice(movie_titles))
            duration = int(movie_duration_map[title])

            completion_ratio = float(np.clip(rng.normal(0.82, 0.18), 0.35, 1.0))
            streamed_for = int(
                max(20, min(duration, round(duration * completion_ratio)))
            )

            event_id = f"{customer_id}|{stream_dt.isoformat()}|{title}|{rng.integers(10_000, 99_999)}"

            watch_rows.append(
                {
                    "customer_id": customer_id,
                    "movie_title": title,
                    "stream_date": stream_dt.isoformat(),
                    "streamed_for": streamed_for,
                    "event_id": event_id,
                }
            )

watch_df = pd.DataFrame(watch_rows)
watch_df.head(), len(watch_df)

(  customer_id       movie_title stream_date  streamed_for  \
 0  8640-SDGKB    Crimson Shadow  2025-11-08           122   
 1  8640-SDGKB   Broken Bridge 2  2025-11-10           119   
 2  8640-SDGKB      Crimson City  2025-11-21           115   
 3  8640-SDGKB   Silent Beacon 5  2025-11-27           172   
 4  8640-SDGKB  Fading Archive 4  2025-11-23           158   
 
                                        event_id  
 0    8640-SDGKB|2025-11-08|Crimson Shadow|95049  
 1   8640-SDGKB|2025-11-10|Broken Bridge 2|18842  
 2      8640-SDGKB|2025-11-21|Crimson City|69868  
 3   8640-SDGKB|2025-11-27|Silent Beacon 5|47671  
 4  8640-SDGKB|2025-11-23|Fading Archive 4|56929  ,
 863285)

In [5]:
# Only about 5-10% of customers will rate movies
rating_customer_share = float(rng.uniform(0.05, 0.10))
n_rating_customers = max(1, int(len(customers_df) * rating_customer_share))
rating_customer_ids = set(
    rng.choice(
        customers_df["customer_id"].to_numpy(), size=n_rating_customers, replace=False
    ).tolist()
)

rated_rows = []
if not watch_df.empty:
    for customer_id in rating_customer_ids:
        cust_watches = watch_df[watch_df["customer_id"] == customer_id]
        if cust_watches.empty:
            continue

        # 1-3 ratings per rating-customer (bounded by available watches)
        k = int(min(len(cust_watches), rng.integers(1, 4)))
        sampled = cust_watches.sample(n=k, random_state=int(rng.integers(0, 1_000_000)))

        for r in sampled.itertuples(index=False):
            # Positively skewed ratings with occasional low scores
            rating = int(rng.choice([1, 2, 3, 4, 5], p=[0.04, 0.08, 0.22, 0.40, 0.26]))
            rated_rows.append(
                {
                    "customer_id": r.customer_id,
                    "movie_title": r.movie_title,
                    "rating": rating,
                }
            )

rated_df = pd.DataFrame(rated_rows).drop_duplicates(
    subset=["customer_id", "movie_title"]
)

actual_raters = rated_df["customer_id"].nunique() if not rated_df.empty else 0
actual_share = (actual_raters / len(customers_df)) if len(customers_df) else 0
print(f"Target rating share: {rating_customer_share:.2%}")
print(
    f"Actual rating customers: {actual_raters} / {len(customers_df)} ({actual_share:.2%})"
)
rated_df.head(), len(rated_df)

Target rating share: 9.94%
Actual rating customers: 271 / 2732 (9.92%)


(  customer_id      movie_title  rating
 0  7605-BDWDC  Velvet Memory 2       4
 1  5832-TRLPB     Lost Frame 5       3
 2  5832-TRLPB    Echo Shadow 3       4
 3  8086-OVPWV      Last Signal       5
 4  8086-OVPWV        Lost Code       3,
 549)

In [6]:
if ENABLE_CLEANUP:
    cleanup_query = """
    MATCH (:Customer)-[w:WATCHED_MOVIE]->(m:Movie)
    WHERE w.event_id IS NOT NULL
    DELETE w
    WITH DISTINCT m
    WHERE NOT (m)<-[:WATCHED_MOVIE]-(:Customer) AND NOT (m)<-[:RATED]-(:Customer)
    DETACH DELETE m
    """

    cleanup_ratings_query = """
    MATCH (:Customer)-[r:RATED]->(:Movie)
    DELETE r
    """

    with driver.session(database=NEO4J_DATABASE) as session:
        session.run(cleanup_ratings_query)
        session.run(cleanup_query)

    print(
        "Cleanup completed: removed prior synthetic WATCHED_MOVIE/RATED relationships and orphan movies."
    )
else:
    print(
        "Cleanup skipped. Set ENABLE_CLEANUP = True to remove prior synthetic enrichment data."
    )

Cleanup completed: removed prior synthetic WATCHED_MOVIE/RATED relationships and orphan movies.


In [7]:
def chunk_list(items, chunk_size=5000):
    for i in range(0, len(items), chunk_size):
        yield items[i : i + chunk_size]


def num_chunks(n_items, chunk_size):
    if n_items == 0:
        return 0
    return (n_items + chunk_size - 1) // chunk_size


create_movie_constraint = "CREATE CONSTRAINT movie_title IF NOT EXISTS FOR (m:Movie) REQUIRE m.title IS UNIQUE"

merge_movies_query = """
UNWIND $rows AS row
MERGE (m:Movie {title: row.title})
SET m.duration = toInteger(row.duration),
    m.year = toInteger(row.year),
    m.rating = toFloat(row.rating)
"""

merge_watch_query = """
UNWIND $rows AS row
MATCH (c:Customer {customer_id: row.customer_id})
MATCH (m:Movie {title: row.movie_title})
MERGE (c)-[w:WATCHED_MOVIE {event_id: row.event_id}]->(m)
SET w.stream_date = date(row.stream_date),
    w.streamed_for = toInteger(row.streamed_for)
"""

merge_rating_query = """
UNWIND $rows AS row
MATCH (c:Customer {customer_id: row.customer_id})
MATCH (m:Movie {title: row.movie_title})
MERGE (c)-[r:RATED]->(m)
SET r.rating = toInteger(row.rating)
"""

movie_rows = movies_df.to_dict(orient="records")
watch_rows = watch_df.to_dict(orient="records")
rating_rows = rated_df.to_dict(orient="records") if not rated_df.empty else []

movie_chunk_size = 2000
watch_chunk_size = 10000
rating_chunk_size = 10000

print(
    f"Preparing writes -> movies: {len(movie_rows)}, watches: {len(watch_rows)}, ratings: {len(rating_rows)}"
)
print(
    f"Chunk plan -> movies: {num_chunks(len(movie_rows), movie_chunk_size)}, "
    f"watches: {num_chunks(len(watch_rows), watch_chunk_size)}, "
    f"ratings: {num_chunks(len(rating_rows), rating_chunk_size)}"
)

with driver.session(database=NEO4J_DATABASE) as session:
    session.run(create_movie_constraint)

    for chunk in tqdm(
        chunk_list(movie_rows, movie_chunk_size),
        total=num_chunks(len(movie_rows), movie_chunk_size),
        desc="Writing Movie nodes",
        unit="chunk",
    ):
        session.run(merge_movies_query, {"rows": chunk})

    for chunk in tqdm(
        chunk_list(watch_rows, watch_chunk_size),
        total=num_chunks(len(watch_rows), watch_chunk_size),
        desc="Writing WATCHED_MOVIE",
        unit="chunk",
    ):
        session.run(merge_watch_query, {"rows": chunk})

    for chunk in tqdm(
        chunk_list(rating_rows, rating_chunk_size),
        total=num_chunks(len(rating_rows), rating_chunk_size),
        desc="Writing RATED",
        unit="chunk",
    ):
        session.run(merge_rating_query, {"rows": chunk})

print(f"Upserted movies: {len(movie_rows)}")
print(f"Upserted WATCHED_MOVIE relationships: {len(watch_rows)}")
print(f"Upserted RATED relationships: {len(rating_rows)}")

Preparing writes -> movies: 400, watches: 863285, ratings: 549
Chunk plan -> movies: 1, watches: 87, ratings: 1


Writing Movie nodes:   0%|          | 0/1 [00:00<?, ?chunk/s]

Writing WATCHED_MOVIE:   0%|          | 0/87 [00:00<?, ?chunk/s]

Writing RATED:   0%|          | 0/1 [00:00<?, ?chunk/s]

Upserted movies: 400
Upserted WATCHED_MOVIE relationships: 863285
Upserted RATED relationships: 549


In [8]:
verification_query = """
MATCH (c:Customer)-[w:WATCHED_MOVIE]->(m:Movie)
RETURN count(DISTINCT c) AS customers_with_watches,
       count(DISTINCT m) AS movies_created,
       count(w) AS watched_rels
"""

verification_ratings = """
MATCH (c:Customer)-[r:RATED]->(:Movie)
RETURN count(DISTINCT c) AS customers_who_rated,
       count(r) AS rating_rels
"""

with driver.session(database=NEO4J_DATABASE) as session:
    v1 = session.run(verification_query).single().data()
    v2 = session.run(verification_ratings).single().data()

print(v1)
print(v2)

{'customers_with_watches': 2732, 'movies_created': 400, 'watched_rels': 863285}
{'customers_who_rated': 271, 'rating_rels': 549}


In [9]:
driver.close()
print("Neo4j driver closed.")

Neo4j driver closed.
